# 02 · Inheritance

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 60 minutes live | [01 · Classes and objects](01_classes_and_objects.ipynb) | [03 · Special methods](03_special_methods.ipynb) |

Very often several classes do almost the same thing and differ in one or two details: several preprocessing steps that all learn something on training data and then apply it, several models that are all evaluated the same way. **Inheritance** lets you write the shared part once, in a base class, and only the differences in each subclass. It is also the backbone of every library you use: scikit-learn's estimators are built from small parent classes.

The theory parts use a small family of scalers. In the hands-on parts you build simple baseline models for the churn data and compare them with the retention team's rule of thumb.

> **Learning goals.** By the end you can:
>
> - write a subclass that inherits behaviour from a base class and overrides one method
> - use `super()` to extend a parent's method instead of copying it
> - read a class hierarchy (`isinstance`, `__mro__`) in your own and in library code
> - judge whether inheritance is the right tool, using the "is-a" test

**Before you start.** Notebook 01 (classes, `__init__`, methods, properties, `@classmethod`).

**In this notebook**

- Theory 1 · Base classes and subclasses
- Theory 2 · Overriding and `super()`
- Hands-on A · Baseline models for churn
- Theory 3 · The MRO, and when not to inherit
- Hands-on B · Reading and combining class hierarchies

---

## Setup

Data for the hands-on parts: the churn table, cleaned as in notebook 01, split into training and test customers, and the `ConfusionMatrix` class from notebook 01. The theory parts do not need any of this.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH)
# TotalCharges is stored as text; to_numeric turns it into numbers (blanks become NaN), fillna(0.0) replaces NaN by 0
customers["TotalCharges"] = pd.to_numeric(customers["TotalCharges"], errors="coerce").fillna(0.0)
# (column == "Yes") gives True/False per row; astype(int) turns that into 1/0
customers["Churn"] = (customers["Churn"] == "Yes").astype(int)

X = customers.drop(columns=["customerID", "Churn"])  # every column except the ID and the label
y = customers["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


class ConfusionMatrix:
    """Hits and misses of a binary classifier (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t, p = np.asarray(y_true), np.asarray(y_pred)  # plain NumPy arrays of 0s and 1s
        # (t == 1) & (p == 0) is True where both hold; .sum() counts the Trues
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def n(self) -> int:
        return self.tn + self.fp + self.fn + self.tp

    @property
    def accuracy(self) -> float:
        return (self.tp + self.tn) / self.n

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if p + r else 0.0


print(f"{len(X_train)} training customers, {len(X_test)} test customers")

---

## Theory 1 · Base classes and subclasses

Think of scalers, the preprocessing steps that put numbers on a common scale. A min-max scaler maps the training range to 0 to 1; a standard scaler subtracts the mean and divides by the standard deviation. They differ in *what* they learn and *how* they transform, but they share the workflow: `fit` learns from training values, `transform` applies what was learned, and `fit_transform` does both in one call. Writing `fit_transform` again in every scaler would be duplicated code.

**Inheritance** puts the shared code into a **base class** (also called parent or superclass). A **subclass** (child) is written `class Child(Parent):` and automatically has every method and attribute the parent defines. The subclass then adds or replaces only what is different.

In [ ]:
class Scaler:
    """Base class: learn statistics on training values, apply them to any values."""

    def __init__(self, name: str | None = None):
        # type(self).__name__ is the name of the object's class, e.g. "MinMaxScaler"
        self.name = name or type(self).__name__

    def fit(self, values: list[float]) -> "Scaler":
        return self  # learns nothing; subclasses that learn something replace this

    def transform(self, values: list[float]) -> list[float]:
        raise NotImplementedError(f"{type(self).__name__} must implement transform()")

    def fit_transform(self, values: list[float]) -> list[float]:
        return self.fit(values).transform(values)

    def describe(self) -> str:
        return f"{self.name}: {type(self).__doc__}"


class MinMaxScaler(Scaler):
    """Maps the training range to [0, 1]."""

    def fit(self, values):
        self.low = min(values)
        self.high = max(values)
        return self

    def transform(self, values):
        span = self.high - self.low
        return [(v - self.low) / span for v in values]


class StandardScaler(Scaler):
    """Subtracts the training mean, divides by the training standard deviation."""

    def fit(self, values):
        self.mean = sum(values) / len(values)
        self.std = (sum((v - self.mean) ** 2 for v in values) / len(values)) ** 0.5
        return self

    def transform(self, values):
        return [(v - self.mean) / self.std for v in values]


monthly_charges = [20.0, 45.0, 70.0, 95.0, 120.0]  # training values
for scaler in [MinMaxScaler(), StandardScaler()]:
    print(scaler.describe())
    print("   ", [round(v, 2) for v in scaler.fit_transform(monthly_charges)])

Neither subclass defines `fit_transform` or `describe`, yet both have them: they are **inherited** from `Scaler`. If we improve `describe` in the base class, every scaler gets the improvement.

Notice how `Scaler.transform` raises `NotImplementedError`. The base class says "every scaler must be able to transform, but I cannot know how". Notebook 04 shows a stricter tool for this (abstract base classes).

### `isinstance` follows the hierarchy

An object of a subclass **is also** an object of its parent class. `isinstance(obj, SomeClass)` answers "is `obj` a `SomeClass`, directly or through a parent?". Code written for a `Scaler` therefore accepts every kind of scaler.

In [ ]:
mm = MinMaxScaler()
print(isinstance(mm, MinMaxScaler))
print(isinstance(mm, Scaler))
print(isinstance(mm, StandardScaler))
print(issubclass(StandardScaler, Scaler))  # the same question about classes

---

## Theory 2 · Overriding and `super()`

When a subclass defines a method with the same name as its parent, the subclass version wins. That is **overriding**. Python looks for a method on the object's own class first and only then on its parents.

Overriding has a powerful consequence. Code in the parent that calls `self.something()` runs the **child's** version of `something`, because `self` is the child object. `fit_transform` is written once in `Scaler`, but `mm.fit_transform(...)` runs `MinMaxScaler.fit` and `MinMaxScaler.transform`.

> **Check your understanding**
>
> ```python
> class Centerer(Scaler):
>     def fit(self, values):
>         self.mean = sum(values) / len(values)
>         return self
>
>     def transform(self, values):
>         return [v - self.mean for v in values]
>
>     def describe(self):
>         return "subtracts the training mean"
>
>
> c = Centerer()
> print(c.fit_transform([1.0, 2.0, 3.0]))
> print(c.describe())
> ```
>
> What do the two lines print?
>
> <details><summary><b>Show answer</b></summary>
>
> `[-1.0, 0.0, 1.0]` and `subtracts the training mean`. `fit_transform` is inherited from `Scaler`, but inside it `self.fit` and `self.transform` are the **subclass's** methods, because `self` is a `Centerer`. `describe` is overridden, so the subclass version runs. This is the central trick of inheritance: shared code in the parent calls methods that each child fills in.
> </details>

In [ ]:
# Predict the output first, then run this cell
class Centerer(Scaler):
    def fit(self, values):
        self.mean = sum(values) / len(values)
        return self

    def transform(self, values):
        return [v - self.mean for v in values]

    def describe(self):
        return "subtracts the training mean"


c = Centerer()
print(c.fit_transform([1.0, 2.0, 3.0]))
print(c.describe())

### `super()`: extend the parent instead of replacing it

Sometimes a subclass wants the parent's behaviour **plus** a little more. New data can fall outside the training range: a min-max scaler then produces values below 0 or above 1. A variant that clips them needs

- its own `__init__`, because it has a parameter (`clip`), but the parent's `__init__` must still run to set `self.name`;
- its own `transform`, but the scaling itself should not be copied from `MinMaxScaler`.

`super()` gives access to the parent's version of a method. The child calls it, then adds its own step.

In [ ]:
class ClippingMinMaxScaler(MinMaxScaler):
    """Like MinMaxScaler, but keeps new values inside [0, 1]."""

    def __init__(self, clip: bool = True):
        super().__init__(name="MinMaxScaler (clipped)" if clip else "MinMaxScaler")
        self.clip = clip

    def transform(self, values):
        scaled = super().transform(values)  # the parent's computation ...
        if self.clip:
            scaled = [min(max(v, 0.0), 1.0) for v in scaled]  # ... plus our extra step
        return scaled


new_month = [10.0, 70.0, 150.0]  # 10 and 150 lie outside the training range 20 to 120
plain = MinMaxScaler().fit(monthly_charges)
clipped = ClippingMinMaxScaler().fit(monthly_charges)
print(plain.name, [round(v, 2) for v in plain.transform(new_month)])
print(clipped.name, [round(v, 2) for v in clipped.transform(new_month)])

> **Check your understanding**
>
> What happens if a subclass defines `__init__` but forgets the `super().__init__(...)` line?
>
> ```python
> class ForgetfulScaler(MinMaxScaler):
>     def __init__(self, clip=True):
>         self.clip = clip
>
>
> ForgetfulScaler().describe()
> ```
>
> <details><summary><b>Show answer</b></summary>
>
> `AttributeError: 'ForgetfulScaler' object has no attribute 'name'`. Defining `__init__` in the subclass overrides the parent's `__init__` completely, so `self.name = ...` never runs. The error only appears later, in `describe`, which makes it confusing. Rule: if a subclass defines `__init__`, it almost always calls `super().__init__(...)` first.
> </details>

In [ ]:
# Predict the output first, then run this cell
class ForgetfulScaler(MinMaxScaler):
    def __init__(self, clip=True):
        self.clip = clip


try:
    print(ForgetfulScaler().describe())
except AttributeError as exc:
    print(f"AttributeError: {exc}")

---

## Hands-on A · Baseline models for churn

Before anyone trains a fancy model, a good data scientist asks: "what would a simple rule achieve?" The retention team already has a rule of thumb ("month-to-month customers are risky"), and any model must beat it to be worth deploying. Rules like this are called **baseline models**. They all need the same evaluation code and differ only in how they predict: exactly the situation from Theory 1.

Here is the base class and two baselines, written with what you just learned. Read the code, run it, then extend the family in the exercises.

In [ ]:
class BaselineModel:
    """Common behaviour of all rule-based baselines."""

    def __init__(self, name: str | None = None):
        self.name = name or type(self).__name__

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "BaselineModel":
        return self  # most rules learn nothing; subclasses that do learn override this

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        raise NotImplementedError(f"{type(self).__name__} must implement predict()")

    def evaluate(self, X: pd.DataFrame, y: pd.Series) -> ConfusionMatrix:
        return ConfusionMatrix.from_predictions(y, self.predict(X))

    def report(self, X: pd.DataFrame, y: pd.Series) -> str:
        cm = self.evaluate(X, y)
        return (
            f"{self.name:<44} flagged={cm.tp + cm.fp:5d}  accuracy={cm.accuracy:.2f}  "
            f"precision={cm.precision:.2f}  recall={cm.recall:.2f}  f1={cm.f1:.2f}"
        )


class MajorityClassModel(BaselineModel):
    """Predicts the most frequent class for everybody."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "MajorityClassModel":
        self.majority_class = int(
            y.mode()[0]
        )  # mode(): the most frequent value(s); [0] takes the first
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return np.full(len(X), self.majority_class)  # an array with len(X) copies of the same value


class ContractRuleModel(BaselineModel):
    """Flags customers whose contract type is in `risky_contracts`."""

    def __init__(self, risky_contracts: tuple[str, ...] = ("Month-to-month",)):
        super().__init__(name=f"contract in {list(risky_contracts)}")
        self.risky_contracts = risky_contracts

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        # isin(...) is True where the value is one of the given ones; astype(int) makes 1/0; to_numpy() a NumPy array
        return X["Contract"].isin(self.risky_contracts).astype(int).to_numpy()


baselines = [
    MajorityClassModel(),
    ContractRuleModel(),
    ContractRuleModel(("Month-to-month", "One year")),
]
for model in baselines:
    print(model.fit(X_train, y_train).report(X_test, y_test))

Two data science lessons in three lines:

- The majority-class model reaches about 73 % accuracy and finds **zero** churners. Accuracy is a misleading score when classes are imbalanced.
- The month-to-month rule finds most churners, at the price of many false alarms. Adding one-year contracts catches almost every churner but costs about 300 extra calls, mostly to customers who would have stayed.

### Exercise A.1

The team suspects that **new customers** churn most. Write `TenureRuleModel(BaselineModel)` that flags customers whose `tenure` (months with the company) is below `max_tenure`. Override only what you need: `fit`, `evaluate` and `report` are inherited. Then compare it with the contract rule.

**What your code should do.**

- `TenureRuleModel(max_tenure=12)` stores the cutoff in `self.max_tenure` and calls `super().__init__` with a readable name that contains the cutoff, for example `"tenure < 12"`.
- `predict(X)` receives a DataFrame with (at least) an integer column `tenure` and returns a NumPy array of 0s and 1s, one per row: 1 if `tenure < max_tenure`, otherwise 0.
- Example: for tenures `[0, 11, 12, 40]` and `max_tenure=12` the result is `array([1, 1, 0, 0])`.

**pandas and NumPy you need.**

- `X["tenure"] < 12`: a Series of `True`/`False`, one per customer.
- `.astype(int)`: turns `True`/`False` into `1`/`0`.
- `.to_numpy()`: converts the Series into a plain NumPy array, which is what models return from `predict`.

**Discuss.** Which rule would you hand to the retention team, the contract rule or the tenure rule, and why? What would you need to know about the cost of a call and the cost of a lost customer to decide?

In [ ]:
# Exercise A.1: your code here


class TenureRuleModel(BaselineModel):
    def __init__(self, max_tenure: int = 12):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise A.1: run after solving it
rule = TenureRuleModel(max_tenure=12)
assert rule.max_tenure == 12
assert isinstance(rule, BaselineModel)
preds = rule.predict(pd.DataFrame({"tenure": [0, 11, 12, 40]}))
assert list(preds) == [1, 1, 0, 0]
assert "12" in rule.name

for model in [ContractRuleModel(), TenureRuleModel(12), TenureRuleModel(6)]:
    print(model.report(X_test, y_test))
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class TenureRuleModel(BaselineModel):
    def __init__(self, max_tenure: int = 12):
        super().__init__(name=f"tenure < {max_tenure}")
        self.max_tenure = max_tenure

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (X["tenure"] < self.max_tenure).astype(int).to_numpy()
```

The tenure rule calls far fewer customers and is more precise, but it misses more churners than the contract rule. Which is better depends on what a call costs compared with a lost customer. We will put numbers on that in notebook 03.
</details>

### Exercise A.2

> If you finish early, try this one.

Why 12 months? Instead of guessing, **learn** the cutoff from the training data. Write `LearnedTenureRuleModel(TenureRuleModel)` that overrides only `fit`. `predict` must stay untouched: it is inherited from `TenureRuleModel`.

**What your code should do.**

- `fit(X, y)` tries every whole number from 1 to 72 as `self.max_tenure`, measures the F1 score on `(X, y)` with the inherited `evaluate`, and keeps the cutoff with the highest F1.
- After `fit`, `self.max_tenure` holds the best cutoff and `self.name` mentions it, for example `"tenure < 23 (learned)"`.
- `fit` returns `self`, so `LearnedTenureRuleModel().fit(X_train, y_train).report(...)` works.

**pandas and NumPy you need.** None: `self.evaluate(X, y).f1` does all the counting for you.

**Discuss.** The cutoff is chosen on the training data and judged on the test data. What would go wrong if you picked it on the test data? And when is a parameter (like `max_tenure`) a better way to add a variation than a new subclass?

In [ ]:
# Exercise A.2: your code here


class LearnedTenureRuleModel(TenureRuleModel):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "LearnedTenureRuleModel":
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise A.2: run after solving it
learned = LearnedTenureRuleModel().fit(X_train, y_train)
# __dict__ lists what a class defines itself (not what it inherits)
assert "predict" not in LearnedTenureRuleModel.__dict__, (
    "predict should be inherited, not redefined"
)
assert 1 <= learned.max_tenure <= 72
assert learned.evaluate(X_train, y_train).f1 >= TenureRuleModel(12).evaluate(X_train, y_train).f1
print(f"learned cutoff: {learned.max_tenure} months")
print(learned.report(X_test, y_test))
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Inside `fit`, loop over the candidates, set `self.max_tenure = candidate`, and call `self.evaluate(X, y).f1`. The inherited `predict` uses whatever `self.max_tenure` is at that moment. Keep the best one at the end.
</details>

<details><summary><b>Show solution</b></summary>

```python
class LearnedTenureRuleModel(TenureRuleModel):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "LearnedTenureRuleModel":
        best_f1, best_cutoff = -1.0, self.max_tenure
        for cutoff in range(1, 73):
            self.max_tenure = cutoff
            f1 = self.evaluate(X, y).f1
            if f1 > best_f1:
                best_f1, best_cutoff = f1, cutoff
        self.max_tenure = best_cutoff
        self.name = f"tenure < {best_cutoff} (learned)"
        return self
```

Note that the cutoff is chosen on the **training** data and judged on the test data. Choosing it on the test data would make the test score optimistic.
</details>

---

## Theory 3 · The MRO, and when not to inherit

### Looking up the chain: the MRO

How does Python decide which method to run? It searches a list of classes called the **method resolution order** (MRO): the object's own class first, then its parent, then the parent's parent, up to `object`, the root of every class in Python. The first class in that list that defines the method wins.

In [ ]:
for cls in ClippingMinMaxScaler.__mro__:
    print(cls.__name__)

For `ClippingMinMaxScaler().fit_transform(...)`, Python looks in `ClippingMinMaxScaler` (not there), then `MinMaxScaler` (not there), then `Scaler` (found). For `transform` it stops at the first class: `ClippingMinMaxScaler` itself.

The MRO is also how you find your way around library code. scikit-learn's estimators have long MROs made of small parent classes; you will read some of them in notebook 06.

In [ ]:
from sklearn.linear_model import LogisticRegression

[cls.__name__ for cls in LogisticRegression.__mro__]

### When not to inherit

Inheritance means "**is a**": a `MinMaxScaler` *is a* `Scaler`, and it can be used everywhere a `Scaler` is expected. Ask that question before you inherit.

- A dataset *is not* a DataFrame. It *has* a DataFrame. Writing `class Dataset(pd.DataFrame)` would expose hundreds of DataFrame methods that make no sense for it, and pandas makes subclassing hard on purpose.
- A model that averages the predictions of two models *is not* both models. It *has* two models.

"Has a" relationships are expressed by storing one object inside another (**composition**). That is often the better tool, and it is a main topic of notebook 08. Inheritance is for sharing behaviour between genuinely similar things, like our scalers or the baselines.

---

## Hands-on B · Reading and combining class hierarchies

Two practical skills: finding out *where* a method really comes from (useful every time you read library code or a confusing error), and deciding between inheritance and composition when you combine baselines.

### Exercise B.1

Write a function `who_defines(cls, method_name)` that walks along the MRO of `cls` and returns the **name** of the first class that defines `method_name` itself. Use it on the scalers, on the baselines and on scikit-learn's `LogisticRegression`.

**What your code should do.**

- `cls` is a class (not an object), `method_name` a string such as `"fit_transform"`.
- Return the class name as a string, for example `who_defines(ClippingMinMaxScaler, "fit_transform")` returns `"Scaler"` and `who_defines(ClippingMinMaxScaler, "transform")` returns `"ClippingMinMaxScaler"`.
- Return `None` if no class in the MRO defines the method.
- Two tools: `cls.__mro__` is the tuple of classes Python searches, in order; `"fit" in SomeClass.__dict__` is `True` only if `SomeClass` defines `fit` itself (inherited methods do not appear in `__dict__`).

**Discuss.** `who_defines(LogisticRegression, "score")` points to a class whose name ends in `Mixin`. What do you think a mixin is for, judging from its name and from what it defines? (Notebook 05 gives the answer.)

In [ ]:
# Exercise B.1: your code here


def who_defines(cls, method_name: str) -> str | None:
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.1: run after solving it
assert who_defines(ClippingMinMaxScaler, "fit_transform") == "Scaler"
assert who_defines(ClippingMinMaxScaler, "fit") == "MinMaxScaler"
assert who_defines(ClippingMinMaxScaler, "transform") == "ClippingMinMaxScaler"
assert who_defines(ContractRuleModel, "report") == "BaselineModel"
assert who_defines(ContractRuleModel, "fly") is None
for name in ["fit", "predict", "score", "get_params"]:
    print(f"LogisticRegression.{name:<10} comes from {who_defines(LogisticRegression, name)}")
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def who_defines(cls, method_name: str) -> str | None:
    for klass in cls.__mro__:
        if method_name in klass.__dict__:
            return klass.__name__
    return None
```

`LogisticRegression` defines only `fit` itself. `predict` comes from `LinearClassifierMixin`, `score` from `ClassifierMixin` and `get_params` from `BaseEstimator`. Each parent contributes one small piece of behaviour.
</details>

### Exercise B.2

> **Challenge**

The team wants a rule that flags a customer if **any** of several rules fires (for example "month-to-month **or** tenure below 6").

1. A colleague proposes `class CombinedRule(ContractRuleModel, TenureRuleModel)`, using **multiple inheritance**: a class that lists more than one parent. Python then searches the parents from left to right (its MRO) and uses the first method it finds. Notebook 05 shows where multiple inheritance is useful (mixins). Explain why this is the wrong tool here. (Consider: how many `predict` methods can the class have, which `__init__` runs, and what about three rules, or two tenure rules with different cutoffs?)
2. Implement `AnyRuleModel(BaselineModel)` with **composition** instead.
3. Is `AnyRuleModel` itself a baseline (is-a), and does it have baselines (has-a)? What does that let you do?

**What your code should do.**

- `AnyRuleModel(rules)` receives a list of baseline objects and keeps them; its name joins the rule names, for example `"contract in ['Month-to-month'] OR tenure < 6"`.
- `fit(X, y)` fits every rule and returns `self`.
- `predict(X)` returns a NumPy array of 0s and 1s with a 1 wherever **at least one** rule predicts 1.

**pandas and NumPy you need.**

- `np.column_stack([a, b, c])`: puts several 1-D arrays of equal length side by side as the columns of a 2-D array (one row per customer, one column per rule).
- `.max(axis=1)`: the largest value in each row, so 1 if any rule said 1. Example: `np.column_stack([[1, 0], [0, 0]]).max(axis=1)` gives `array([1, 0])`.

**Discuss.** Where else have you seen an object that is one of the things it contains (think of scikit-learn's `Pipeline`, or folders on your disk)? In which situation would you still choose inheritance over composition?

In [ ]:
# Exercise B.2: your code here


class AnyRuleModel(BaselineModel):
    def __init__(self, rules: list[BaselineModel]):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.2: run after solving it
combo = AnyRuleModel([ContractRuleModel(), TenureRuleModel(6)])
combo.fit(X_train, y_train)
toy = pd.DataFrame({"Contract": ["Two year", "Month-to-month", "Two year"], "tenure": [3, 40, 40]})
assert list(combo.predict(toy)) == [1, 1, 0]
nested = AnyRuleModel([combo, TenureRuleModel(2)])
assert list(nested.fit(X_train, y_train).predict(toy)) == [1, 1, 0]
print(combo.report(X_test, y_test))
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class AnyRuleModel(BaselineModel):
    def __init__(self, rules: list[BaselineModel]):
        super().__init__(name=" OR ".join(rule.name for rule in rules))
        self.rules = rules

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "AnyRuleModel":
        for rule in self.rules:
            rule.fit(X, y)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        votes = np.column_stack([rule.predict(X) for rule in self.rules])
        return votes.max(axis=1)
```

1. With multiple inheritance a class still has exactly one `predict`: Python picks the first one in the MRO (`ContractRuleModel.predict`) and the tenure rule is silently ignored. Only one parent's `__init__` runs unless every class cooperates through `super()`, which these do not. And inheritance is fixed when the class is written: "two tenure rules with different cutoffs" or "any three rules" would each need a new class.
2. With composition, the rules are ordinary objects chosen at run time: any number, any mix, any parameters.
3. Both. `AnyRuleModel` *is a* `BaselineModel` (so it inherits `evaluate` and `report` and can be used wherever a baseline is expected) and it *has* baselines. Because it is one of the things it contains, combinations can be nested, as in the check cell. This is the **Composite** pattern; scikit-learn's `Pipeline` and `VotingClassifier` follow the same idea.
</details>

---

## Summary

- **Inheritance** lets a subclass reuse everything its base class defines: `class Child(Parent):`.
- **Overriding** replaces a parent's method with a more specific one. Shared code in the parent can call methods that each child provides (`fit_transform` calls `self.fit`, `evaluate` calls `self.predict`).
- `super()` runs the parent's version of a method, so a child can extend it instead of copying it. Forget `super().__init__(...)` and attributes go missing.
- Python finds methods along the **MRO**; `isinstance`, `__mro__` and `__dict__` show you the hierarchy, in your code and in libraries.
- Use inheritance only for genuine **is-a** relationships. For **has-a**, store objects inside objects (composition).
- Data science lesson along the way: always compare against simple baselines, and do not trust accuracy on imbalanced data.

**Common pitfalls:** defining `__init__` in a subclass without calling `super().__init__`; deep hierarchies where nobody remembers which parent provides what; inheriting from a class (like `pd.DataFrame`) just to reuse a few of its methods.

## Going further

- scikit-learn's own baselines: `sklearn.dummy.DummyClassifier` (we read its source in notebook 06).
- scikit-learn has real scalers (`MinMaxScaler`, `StandardScaler` in `sklearn.preprocessing`) that work on whole tables; ours work on lists to keep the theory small.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [The Python Tutorial, section 9.5: Inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance).
- [The Python 2.3 Method Resolution Order](https://docs.python.org/3/howto/mro.html), the official explanation of the MRO algorithm (advanced).
- scikit-learn's baseline models: [`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html).

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Inheritance and Composition, a Python OOP Guide](https://realpython.com/inheritance-composition-python/) (article, intermediate): when to use which, with a longer worked example.
- [Real Python: Supercharge Your Classes With Python super()](https://realpython.com/python-super/) (article, intermediate).
- [Raymond Hettinger, *Python's super() considered super!*](https://rhettinger.wordpress.com/2011/05/26/super-considered-super/) (blog post, advanced): how `super()` really works with multiple inheritance.

---

[← 01 · Classes and objects](01_classes_and_objects.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [03 · Special methods →](03_special_methods.ipynb)